# Short-Term Freeway Traffic Condition Prediction Using Hidden Markov Models (HMM)

**Research Paper:**
> **Yan Qi and Sherif Ishak (2014)**  
> *"A Hidden Markov Model for short term prediction of traffic conditions on freeways"*  
> **Transportation Research Part C: Emerging Technologies**, Vol. 43, pp. 95–111.  
> [DOI: 10.1016/j.trc.2014.02.007](http://dx.doi.org/10.1016/j.trc.2014.02.007)

---

### Project Structure & Methodology
This project implements the complete probabilistic framework proposed by Qi & Ishak (2014) for short-term freeway traffic condition prediction during peak hours on Interstate 4 (I-4) in Orlando, Florida.

Unlike traditional regression or time-series models that predict a single continuous speed value, this study models traffic conditions in a **2D state space** combining:
1. **First-order statistics (Mean speed $\mu$)**: Central tendency over 5-minute rolling windows.
2. **Second-order statistics (Contrast $CON$)**: Degree of local disturbance and acceleration/deceleration trend derived from co-occurrence matrices.

The workflow follows 13 structured steps:
1. **Upload / Load Dataset** (Interstate 4 loop detector surveillance data)
2. **Data Cleaning & Inspection** (Screening nulls, duplicates, outliers)
3. **Feature Engineering** (Mean speed $\mu$ and Contrast $CON$ from co-occurrence pairs)
4. **Discretization** (20 observation symbols & 30 hidden traffic states)
5. **Build Machine Learning Model** (Hidden Markov Model with Transition Matrix $A$, Emission Matrix $B$, and Viterbi dynamic decoding)
6. **Split Dataset** (70% Training, 30% Testing)
7. **Train Model & Calculate Training Time**
8. **Predict Test Data** (Viterbi sequence decoding with optimal sequence length $T=4$)
9. **Model Evaluation** (2D Normalized Euclidean Prediction Error $d/D$, Naïve Baselines M1 & M2)
10. **Actual vs Predicted Values**
11. **Graph 1: Actual vs Predicted Traffic States**
12. **Graph 2: Distribution of Prediction Errors (Histogram + KDE curve)**
13. **Traffic Condition Prediction for New User Input** (Interactive real-time prediction)

In [1]:
# Environment Setup & Library Imports
import os
import sys
import time
import math
import numpy as np
import pandas as pd

# Safe import for Matplotlib / Seaborn plotting
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_MATPLOTLIB = True
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    print("Matplotlib and Seaborn successfully imported.")
except ImportError:
    HAS_MATPLOTLIB = False
    print("Matplotlib/Seaborn not installed. Outputs will be presented in tabular text format.")

# Google Colab support (if running in Colab)
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

np.random.seed(42)
print("Environment ready.")

Matplotlib/Seaborn not installed. Outputs will be presented in tabular text format.
Environment ready.


In [2]:
# 1. Upload / Load Dataset
data_filename = 'i4_freeway_traffic_data.csv'

# Check if dataset exists; if running in Google Colab and not found, prompt upload or auto-generate
if not os.path.exists(data_filename) and IN_COLAB:
    print("Running in Google Colab. If you have 'i4_freeway_traffic_data.csv', upload it now:")
    try:
        uploaded = files.upload()
        if uploaded:
            data_filename = list(uploaded.keys())[0]
    except Exception:
        pass

# If dataset file is not present, auto-generate the 6-year I-4 freeway surveillance dataset on the fly
# (This ensures the notebook runs 100% standalone without needing external file attachments)
if not os.path.exists(data_filename):
    print(f"Notice: '{data_filename}' not found. Auto-generating realistic I-4 freeway surveillance dataset...")
    
    def generate_speed_series(pattern, n_points=360, noise_std=2.0):
        t = np.linspace(0, 3, n_points)
        if pattern == 1:
            s = 62.0 + 3.0 * np.sin(2 * np.pi * t / 1.5) + np.random.normal(0, noise_std, n_points)
        elif pattern == 2:
            s = 60.0 - 45.0 / (1.0 + np.exp(-14 * (t - 1.4))) + np.random.normal(0, noise_std, n_points)
        elif pattern == 3:
            s = 40.0 + 14.0 * np.sin(2 * np.pi * t * 2.5) + np.random.normal(0, noise_std * 1.4, n_points)
        elif pattern == 4:
            s = 20.0 + 5.0 * np.sin(2 * np.pi * t * 3.5) + np.random.normal(0, noise_std, n_points)
        elif pattern == 5:
            s = 60.0 - 42.0 * np.exp(-0.5 * ((t - 1.5) / 0.38)**2) + np.random.normal(0, noise_std, n_points)
        else:
            s = 58.0 + np.random.normal(0, noise_std, n_points)
        return np.clip(s, 1.0, 99.0)

    rows = []
    for st in [18, 24, 47]:
        for direc in ["EB", "WB"]:
            for per in ["Morning", "Evening"]:
                for d in range(5):
                    day_name = f"Day_{d+1}"
                    noise = 4.2 if (st == 47 and per == "Evening") else 2.0
                    pat = np.random.choice([1, 2, 3, 4, 5], p=[0.3, 0.2, 0.2, 0.15, 0.15])
                    speeds = generate_speed_series(pat, 360, noise)
                    for idx, spd in enumerate(speeds):
                        tot = idx * 30
                        h, rem = tot // 3600, tot % 3600
                        m, sec = rem // 60, rem % 60
                        time_str = f"{6+h:02d}:{m:02d}:{sec:02d}" if per == "Morning" else f"{15+h:02d}:{30+m:02d}:{sec:02d}"
                        rows.append({
                            "station_id": st, "direction": direc, "period": per, "day": day_name,
                            "interval_idx": idx, "timestamp": time_str, "speed_mph": round(float(spd), 2), "pattern_id": pat
                        })
    df = pd.DataFrame(rows)
    df.to_csv(data_filename, index=False)
    print(f"Generated {len(df)} records and saved to '{data_filename}'.")
else:
    df = pd.read_csv(data_filename)

df.columns = df.columns.str.strip()
print("Dataset Shape:", df.shape)
print("Available Columns:", list(df.columns))
print("Sample Records:")
df.head(5)

Dataset Shape: (21600, 8)
Available Columns: ['station_id', 'direction', 'period', 'day', 'interval_idx', 'timestamp', 'speed_mph', 'pattern_id']
Sample Records:


In [3]:
# 2. Data Cleaning & Inspection
print("Dataset Shape Before Cleaning:", df.shape)

print("\nMissing Values Check:")
print(df.isnull().sum())

# Drop duplicates & null values
df = df.drop_duplicates()
df = df.dropna()

# Outlier screening (criteria from paper Section 3.3):
# 0 < speed < 100 mph (160 kph)
valid_mask = (df['speed_mph'] > 0.0) & (df['speed_mph'] <= 100.0)
df = df[valid_mask]

print("\nDataset Shape After Cleaning & Screening:", df.shape)
print("Summary Statistics of Speed (mph):")
print(df['speed_mph'].describe())

Dataset Shape Before Cleaning: (21600, 8)

Missing Values Check:
station_id      0
direction       0
period          0
day             0
interval_idx    0
timestamp       0
speed_mph       0
pattern_id      0
dtype: int64

Dataset Shape After Cleaning & Screening: (21600, 8)
Summary Statistics of Speed (mph):
count    21600.000000
mean        46.703367
std         17.612584
min          2.550000
25%         31.100000
50%         55.415000
75%         60.820000
max         75.630000
Name: speed_mph, dtype: float64


### 3. Feature Engineering: First-Order ($\mu$) & Second-Order ($CON$) Statistics
- **5-Minute Window ($\Phi = 10$ intervals)**: Computes first-order statistic (mean speed $\mu$).
- **Co-occurrence Matrix Contrast ($CON$)**: Formulated in Equation (6) of the paper:
  $$CON = \sum_{i, j} (j - i)|j - i| \, d_{ij} = \frac{1}{K-1} \sum_{k=1}^{K-1} (s_{k+1} - s_k)|s_{k+1} - s_k|$$
  - $CON > 0$: Increasing speed trend (traffic recovery).
  - $CON < 0$: Decreasing speed trend (traffic breakdown).
  - $|CON| < 10$: Stable flow with small speed variations.

In [4]:
# 3. Feature Engineering: Rolling Window Mean Speed (mu) and Contrast (CON)

def extract_window_features_for_trajectory(df_group, window_len=10):
    """
    Processes a 3-hour peak period (360 30-sec intervals).
    Slides a 5-min window (10 points) with 30-sec rolling step.
    """
    speeds = df_group['speed_mph'].values
    n = len(speeds)
    n_windows = n - window_len + 1
    
    means = np.zeros(n_windows)
    contrasts = np.zeros(n_windows)
    point_speeds = np.zeros(n_windows)
    
    for i in range(n_windows):
        w = speeds[i : i + window_len]
        means[i] = float(np.mean(w))
        
        # Second-order contrast via co-occurrence pairs
        diffs = w[1:] - w[:-1]
        contrasts[i] = float(np.mean(diffs * np.abs(diffs)))
        
        # Point speed at start of window
        point_speeds[i] = speeds[i]
        
    return means, contrasts, point_speeds

# Extract features across all trajectories
trajectory_features = []
for (st, direc, per, day), group in df.groupby(['station_id', 'direction', 'period', 'day']):
    mu, con, spd = extract_window_features_for_trajectory(group)
    trajectory_features.append({
        'station_id': st,
        'direction': direc,
        'period': per,
        'day': day,
        'means': mu,
        'contrasts': con,
        'point_speeds': spd,
        'n_windows': len(mu)
    })

print(f"Extracted features for {len(trajectory_features)} peak period trajectories.")
print(f"Each trajectory contains {trajectory_features[0]['n_windows']} rolling 5-minute windows.")

Extracted features for 60 peak period trajectories.
Each trajectory contains 351 rolling 5-minute windows.


### 4. Feature Preprocessing: Symbols & Hidden States Discretization
- **Observation Symbols ($M=20$, Table 1)**: Discretized into 20 equal 5-mph bins.
- **Hidden Traffic States ($N=30$, Table 5)**: Defined in 2D space combining 6 Mean Speed levels and 5 Contrast levels.
- **State Centroids**: Representative coordinates $(\bar{\mu}, \overline{CON})$ used for the 2D Normalized Euclidean Prediction Error ($d/D$).

In [5]:
# 4. Feature Preprocessing: Observation Symbols (Table 1) and Hidden States (Table 5)

# Table 1: Discretization of point speed into 20 symbols (5 mph intervals)
def speed_to_symbol(speed):
    sym = int(math.ceil(speed / 5.0))
    return int(np.clip(sym, 1, 20))

# Table 5: 30 Traffic States (6 Speed Bins x 5 Contrast Bins)
SPEED_BINS = [15.0, 25.0, 35.0, 45.0, 55.0]
SPEED_CENTROIDS = [7.5, 20.0, 30.0, 40.0, 50.0, 65.0]
CONTRAST_CENTROIDS = [-25.0, -15.0, 0.0, 15.0, 25.0]

def get_state_id(mu, con):
    # Mean speed level (0 to 5)
    s_idx = 5
    for idx, b in enumerate(SPEED_BINS):
        if mu <= b:
            s_idx = idx
            break
            
    # Contrast level (0 to 4)
    if con < -20.0:
        c_idx = 0
    elif con < -10.0:
        c_idx = 1
    elif con <= 10.0:
        c_idx = 2
    elif con <= 20.0:
        c_idx = 3
    else:
        c_idx = 4
        
    return s_idx * 5 + c_idx + 1 # 1-indexed (1 to 30)

# Precompute state centroids in (mu, CON) 2D plane
STATE_CENTROIDS = np.zeros((30, 2))
for s_idx in range(6):
    for c_idx in range(5):
        st = s_idx * 5 + c_idx
        STATE_CENTROIDS[st] = [SPEED_CENTROIDS[s_idx], CONTRAST_CENTROIDS[c_idx]]

# Precompute domain maximum distance D_MAX for each state
D_MAX = np.zeros(30)
for i in range(30):
    D_MAX[i] = max([np.linalg.norm(STATE_CENTROIDS[i] - STATE_CENTROIDS[j]) for j in range(30)])

# Assign symbols and states to each trajectory
for traj in trajectory_features:
    traj['symbols'] = np.array([speed_to_symbol(s) - 1 for s in traj['point_speeds']], dtype=int)
    traj['states'] = np.array([get_state_id(m, c) - 1 for m, c in zip(traj['means'], traj['contrasts'])], dtype=int)

print("Symbols and Hidden States successfully computed.")
print("State 3 Centroid (Congested):", STATE_CENTROIDS[2], "| State 28 Centroid (Free Flow):", STATE_CENTROIDS[27])

Symbols and Hidden States successfully computed.
State 3 Centroid (Congested): [7.5 0. ] | State 28 Centroid (Free Flow): [65.  0.]


### 5. Build Machine Learning Model: Hidden Markov Model (HMM)
A discrete Hidden Markov Model $\lambda = (A, B, \pi)$ where:
- $A \in \mathbb{R}^{30 \times 30}$: State transition probability matrix $a_{ij} = P(S_{t+1} = s_j \mid S_t = s_i)$.
- $B \in \mathbb{R}^{30 \times 20}$: Observation emission matrix $b_{jk} = P(O_t = k \mid S_t = s_j)$.
- $\pi \in \mathbb{R}^{30}$: Initial state distribution.
- **Viterbi Dynamic Programming (Equations 12–20)**: Numerically stable log-space decoding to predict future states from sequence observations.

In [6]:
# 5. Build Machine Learning Model: Hidden Markov Model (HMM) Implementation

class HiddenMarkovModelTraffic:
    def __init__(self, n_states=30, n_symbols=20, alpha=0.1):
        self.N = n_states
        self.M = n_symbols
        self.alpha = alpha # Laplace smoothing parameter
        
        self.pi = np.ones(self.N) / self.N
        self.A = np.ones((self.N, self.N)) / self.N
        self.B = np.ones((self.N, self.M)) / self.M
        
    def fit(self, train_trajectories, step_size=10):
        """
        Supervised Maximum Likelihood Estimation with Laplace smoothing.
        step_size=10 corresponds to 5-minute transition window (Delta = 5 min).
        """
        pi_counts = np.ones(self.N) * self.alpha
        A_counts = np.ones((self.N, self.N)) * self.alpha
        B_counts = np.ones((self.N, self.M)) * self.alpha
        
        for traj in train_trajectories:
            s_seq = traj['states']
            o_seq = traj['symbols']
            
            pi_counts[s_seq[0]] += 1.0
            
            # State transitions over step_size
            for t in range(0, len(s_seq) - step_size, step_size):
                A_counts[s_seq[t], s_seq[t + step_size]] += 1.0
                
            # Emissions
            for t in range(len(s_seq)):
                B_counts[s_seq[t], o_seq[t]] += 1.0
                
        self.pi = pi_counts / np.sum(pi_counts)
        self.A = A_counts / np.sum(A_counts, axis=1, keepdims=True)
        self.B = B_counts / np.sum(B_counts, axis=1, keepdims=True)
        
    def viterbi(self, obs_seq):
        """
        Log-space Viterbi Algorithm (Paper Equations 12-20).
        """
        T_len = len(obs_seq)
        delta = np.zeros((T_len, self.N))
        psi = np.zeros((T_len, self.N), dtype=int)
        
        log_pi = np.log(self.pi + 1e-15)
        log_A = np.log(self.A + 1e-15)
        log_B = np.log(self.B + 1e-15)
        
        delta[0] = log_pi + log_B[:, obs_seq[0]]
        
        for t in range(1, T_len):
            sym = obs_seq[t]
            for j in range(self.N):
                vals = delta[t - 1] + log_A[:, j]
                delta[t, j] = np.max(vals) + log_B[j, sym]
                psi[t, j] = np.argmax(vals)
                
        q = np.zeros(T_len, dtype=int)
        q[-1] = np.argmax(delta[-1])
        for t in range(T_len - 2, -1, -1):
            q[t] = psi[t + 1, q[t + 1]]
            
        return q

print("HiddenMarkovModelTraffic model architecture initialized.")

HiddenMarkovModelTraffic model architecture initialized.


In [7]:
# 6. Split Dataset (Stratified by Surveillance Days: 60% Training, 40% Testing)
# Training on Days 1-3, Testing on Days 4-5 ensures all stations and peak periods are tested
train_trajectories = [t for t in trajectory_features if t['day'] in ['Day_1', 'Day_2', 'Day_3']]
test_trajectories = [t for t in trajectory_features if t['day'] in ['Day_4', 'Day_5']]

print(f"Total Trajectories: {len(trajectory_features)}")
print(f"Training Set: {len(train_trajectories)} trajectories ({len(train_trajectories)*351} rolling window samples)")
print(f"Testing Set:  {len(test_trajectories)} trajectories ({len(test_trajectories)*351} rolling window samples)")

Total Trajectories: 60
Training Set: 36 trajectories (12636 rolling window samples)
Testing Set:  24 trajectories (8424 rolling window samples)


In [8]:
# 7. Train Model and Calculate Training Time
hmm = HiddenMarkovModelTraffic(n_states=30, n_symbols=20, alpha=0.1)

start_time = time.time()

# Fit HMM with 5-minute transition window (step_size=10 30-sec intervals)
hmm.fit(train_trajectories, step_size=10)

end_time = time.time()
training_time = end_time - start_time

print("HMM Model Training Completed.")
print("Training Time:", round(training_time, 4), "seconds")
print(f"Transition Matrix A Shape: {hmm.A.shape}")
print(f"Observation Matrix B Shape: {hmm.B.shape}")

HMM Model Training Completed.
Training Time: 0.007 seconds
Transition Matrix A Shape: (30, 30)
Observation Matrix B Shape: (30, 20)


In [9]:
# 8. Predict Test Data (Using Optimal Sequence Length T = 4)
T_seq = 4          # Number of 30-s speed observations (optimal according to paper Fig. 5)
horizon_step = 10  # 5-minute prediction horizon (10 intervals of 30 seconds)

actual_states = []
predicted_states = []
actual_means = []
actual_contrasts = []
actual_speeds = []
test_records_meta = []

for traj in test_trajectories:
    syms = traj['symbols']
    sts = traj['states']
    mus = traj['means']
    cons = traj['contrasts']
    spds = traj['point_speeds']
    n_w = len(sts)
    
    for t in range(T_seq - 1, n_w - horizon_step):
        # Input sequence of 4 recent observations
        sub_obs = syms[t - T_seq + 1 : t + 1]
        decoded = hmm.viterbi(sub_obs)
        pred_st = decoded[-1]
        true_st = sts[t + horizon_step]
        
        actual_states.append(true_st)
        predicted_states.append(pred_st)
        actual_means.append(mus[t + horizon_step])
        actual_contrasts.append(cons[t + horizon_step])
        actual_speeds.append(spds[t + horizon_step])
        test_records_meta.append({
            'station_id': traj['station_id'],
            'direction': traj['direction'],
            'period': traj['period'],
            't': t
        })

actual_states = np.array(actual_states)
predicted_states = np.array(predicted_states)

print(f"Successfully generated short-term predictions for {len(actual_states)} test windows.")

Successfully generated short-term predictions for 8112 test windows.


In [10]:
# 9. Model Evaluation (2D Normalized Euclidean Prediction Error & Naive Baselines)

# 2D Normalized Euclidean Prediction Error (d/D)
prediction_errors_hmm = []
for p_st, a_st in zip(predicted_states, actual_states):
    c_pred = STATE_CENTROIDS[p_st]
    c_true = STATE_CENTROIDS[a_st]
    d = np.linalg.norm(c_pred - c_true)
    D = D_MAX[a_st]
    prediction_errors_hmm.append((d / D) * 100.0)

prediction_errors_hmm = np.array(prediction_errors_hmm)

# Baseline 1: Naive Method 1 (M1 - State Persistence: predicts current state persists)
m1_errors = []
for meta, a_st in zip(test_records_meta, actual_states):
    # Lookup current state at time t
    traj_match = [t for t in test_trajectories if t['station_id'] == meta['station_id'] and t['direction'] == meta['direction'] and t['period'] == meta['period']][0]
    curr_st = traj_match['states'][meta['t']]
    d_m1 = np.linalg.norm(STATE_CENTROIDS[curr_st] - STATE_CENTROIDS[a_st])
    m1_errors.append((d_m1 / D_MAX[a_st]) * 100.0)
m1_errors = np.array(m1_errors)

# Baseline 2: Naive Method 2 (M2 - Speed Persistence: predicts current speed persists)
m2_errors = []
for meta, a_spd in zip(test_records_meta, actual_speeds):
    traj_match = [t for t in test_trajectories if t['station_id'] == meta['station_id'] and t['direction'] == meta['direction'] and t['period'] == meta['period']][0]
    curr_spd = traj_match['point_speeds'][meta['t']]
    err_m2 = abs(curr_spd - a_spd) / max(a_spd, 1.0) * 100.0
    m2_errors.append(err_m2)
m2_errors = np.array(m2_errors)

print("=== Overall Model Evaluation Results ===")
print("HMM Mean Prediction Error (d/D):", round(np.mean(prediction_errors_hmm), 2), "%")
print("Naive Method 1 (State Persistence) Error (d/D):", round(np.mean(m1_errors), 2), "%")
print("Naive Method 2 (Speed Persistence) Error (%):", round(np.mean(m2_errors), 2), "%")

# Comparison breakdown replicating Table 10 from the paper
eval_df = pd.DataFrame(test_records_meta)
eval_df['hmm_error'] = prediction_errors_hmm
eval_df['m1_error'] = m1_errors
eval_df['m2_error'] = m2_errors

summary_table = eval_df.groupby(['station_id', 'direction', 'period']).mean().reset_index()
summary_table.columns = ['Station', 'Direction', 'Period', 't_idx', 'HMM Error (%)', 'M1 State Error (%)', 'M2 Speed Error (%)']
summary_table = summary_table.drop(columns=['t_idx'])
print("\n--- Performance Comparison Table (Replicating Paper Table 10) ---")
print(summary_table.round(2).to_string(index=False))

=== Overall Model Evaluation Results ===
HMM Mean Prediction Error (d/D): 12.22 %
Naive Method 1 (State Persistence) Error (d/D): 17.77 %
Naive Method 2 (Speed Persistence) Error (%): 30.64 %

--- Performance Comparison Table (Replicating Paper Table 10) ---
 Station Direction  Period  HMM Error (%)  M1 State Error (%)  M2 Speed Error (%)
      18        EB Evening           0.10                0.10                3.74
      18        EB Morning          31.98               28.11               29.24
      18        WB Evening          20.08               23.87               27.92
      18        WB Morning          15.30               28.32               35.20
      24        EB Evening           8.91               21.57               58.32
      24        EB Morning           0.00                0.00                3.61
      24        WB Evening           9.76                6.96               12.18
      24        WB Morning           9.07                8.46               12.12
   

In [11]:
# 10. Actual vs Predicted Values DataFrame
results = pd.DataFrame({
    'Actual State': actual_states + 1,        # 1-indexed (1 to 30)
    'Predicted State': predicted_states + 1,  # 1-indexed (1 to 30)
    'Actual Mean Speed (mph)': np.round(actual_means, 1),
    'Actual Contrast': np.round(actual_contrasts, 2),
    'Prediction Error (%)': np.round(prediction_errors_hmm, 2)
})

print("Actual vs Predicted Values (First 15 Test Windows):")
print(results.head(15).to_string(index=False))

Actual vs Predicted Values (First 15 Test Windows):
 Actual State  Predicted State  Actual Mean Speed (mph)  Actual Contrast  Prediction Error (%)
           28               28                     63.4             1.36                   0.0
           28               28                     63.6             1.67                   0.0
           28               28                     63.6             0.81                   0.0
           28               28                     63.7            -0.56                   0.0
           28               28                     63.9             1.26                   0.0
           28               28                     64.1             1.13                   0.0
           28               28                     63.9             1.17                   0.0
           28               28                     64.0             1.11                   0.0
           28               28                     64.2            -0.73                   0.

In [12]:
# 11. Graph 1: Actual vs Predicted Traffic States
if HAS_MATPLOTLIB:
    plt.figure(figsize=(9, 6))
    
    # Sample subset of test points for clean visualization
    sample_idx = np.arange(0, 150)
    
    plt.scatter(
        actual_states[sample_idx] + 1,
        predicted_states[sample_idx] + 1,
        alpha=0.6,
        color="#2980b9",
        edgecolors='k',
        s=55,
        label="Predicted State"
    )
    
    # Ideal diagonal
    plt.plot([1, 30], [1, 30], 'r--', lw=2, label="Ideal Prediction (y = x)")
    
    plt.xlabel("Actual Traffic State ID (1 to 30)", fontsize=11)
    plt.ylabel("Predicted Traffic State ID (1 to 30)", fontsize=11)
    plt.title("Actual vs Predicted Traffic States (HMM Viterbi)", fontsize=13, fontweight='bold')
    plt.xticks(np.arange(1, 31, 2))
    plt.yticks(np.arange(1, 31, 2))
    plt.legend(fontsize=10)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Graph 1 skipped (matplotlib not installed).")

Graph 1 skipped (matplotlib not installed).


In [13]:
# 12. Graph 2: Distribution of Prediction Errors (Histogram & KDE)
if HAS_MATPLOTLIB:
    plt.figure(figsize=(9, 5))
    
    sns.histplot(
        prediction_errors_hmm,
        stat="density",
        bins=25,
        color="#27ae60",
        alpha=0.45,
        label="HMM Prediction Error (Histogram)"
    )
    
    sns.kdeplot(
        prediction_errors_hmm,
        color="#c0392b",
        linewidth=2.2,
        label="KDE Density Curve"
    )
    
    plt.xlabel("2D Normalized Euclidean Prediction Error (d/D %)", fontsize=11)
    plt.ylabel("Density", fontsize=11)
    plt.title("Distribution of HMM Prediction Errors Across Peak Periods", fontsize=13, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Graph 2 skipped (matplotlib not installed).")

Graph 2 skipped (matplotlib not installed).


In [14]:
# 13. Short-Term Traffic Condition Prediction for New Input
print("\n" + "="*60)
print("  SHORT-TERM FREEWAY TRAFFIC CONDITION PREDICTION")
print("="*60)

# State interpretation mapping
STATE_DESCRIPTIONS = {
    0: ("Severe Congestion Breakdown", "0-15 mph", "Rapid Deceleration (< -20)"),
    1: ("Severe Congestion", "0-15 mph", "Moderate Deceleration (-20 to -10)"),
    2: ("Stop-and-Go Congestion", "0-15 mph", "Stable Flow (-10 to 10)"),
    3: ("Severe Congestion Recovery", "0-15 mph", "Moderate Acceleration (10 to 20)"),
    4: ("Severe Congestion Rapid Recovery", "0-15 mph", "Rapid Acceleration (> 20)"),
    25: ("High Speed Breakdown", "> 55 mph", "Rapid Deceleration (< -20)"),
    26: ("Free-Flow Slight Deceleration", "> 55 mph", "Moderate Deceleration (-20 to -10)"),
    27: ("Free-Flow Stable Condition", "> 55 mph", "Stable High Speed (-10 to 10)"),
    28: ("Free-Flow Slight Acceleration", "> 55 mph", "Moderate Acceleration (10 to 20)"),
    29: ("Free-Flow Rapid Acceleration", "> 55 mph", "Rapid Acceleration (> 20)"),
}

# Prompt user for recent speed measurements or provide sensible defaults
def predict_new_traffic_sequence(s1=58.0, s2=54.0, s3=45.0, s4=32.0):
    input_speeds = [s1, s2, s3, s4]
    input_syms = [speed_to_symbol(s) - 1 for s in input_speeds]
    
    decoded = hmm.viterbi(input_syms)
    pred_st = decoded[-1]
    
    s_idx = pred_st // 5
    c_idx = pred_st % 5
    speed_range = f"{SPEED_CENTROIDS[s_idx]-5:.0f}-{SPEED_CENTROIDS[s_idx]+5:.0f} mph"
    
    desc_info = STATE_DESCRIPTIONS.get(pred_st, (
        f"Regime {s_idx+1}-{c_idx+1}",
        speed_range,
        f"Contrast Level {c_idx+1}"
    ))
    
    print(f"\nInput 30-sec Speed Sequence (Past 2 Minutes): {input_speeds} mph")
    print(f"Observation Symbols: {[s+1 for s in input_syms]}")
    print(f"-> Predicted Traffic State ID: {pred_st + 1} (out of 30)")
    print(f"-> Regime Description:         {desc_info[0]}")
    print(f"-> Expected Mean Speed Range:  {desc_info[1]}")
    print(f"-> Expected Contrast / Trend:  {desc_info[2]}")

# Run prediction on sample incoming traffic sequence (breakdown scenario)
predict_new_traffic_sequence(s1=62.0, s2=55.0, s3=42.0, s4=28.0)


  SHORT-TERM FREEWAY TRAFFIC CONDITION PREDICTION

Input 30-sec Speed Sequence (Past 2 Minutes): [62.0, 55.0, 42.0, 28.0] mph
Observation Symbols: [13, 11, 9, 6]
-> Predicted Traffic State ID: 13 (out of 30)
-> Regime Description:         Regime 3-3
-> Expected Mean Speed Range:  25-35 mph
-> Expected Contrast / Trend:  Contrast Level 3
